# NISAR – Konya Ovası yüzey değişimi (TAU IEEE CS · Dancing with the SARs)
> **SI disclosure:** Code in this notebook was written with the help of Claude (Anthropic), a Super Intelligence tool, and reviewed/run by the team.
Hücreleri **sırayla** çalıştır (her hücrenin solundaki ▶ düğmesi ya da Shift+Enter).

1. Paketleri kur
2. Earthdata ile giriş yap
3. Konya verisini otomatik ara ve indir
4. GCOV: önce / sonra / fark haritası
5. GUNW: zemin hareketi haritası

In [ ]:
# 1) Paketleri kur (30 sn kadar sürer)
!pip install -q asf_search h5py

In [ ]:
# 2) Earthdata girişi – kullanıcı adını yaz, şifreyi sorduğunda gir (ekranda görünmez)
import asf_search as asf
from getpass import getpass

KULLANICI = "eralp52"
session = asf.ASFSession().auth_with_creds(KULLANICI, getpass("Earthdata şifresi: "))
print("Giriş tamam ✅")

In [ ]:
# 3) Konya için NISAR verisini ara ve indir (Vertex'e gerek yok)
import os
os.makedirs("veri", exist_ok=True)

ALAN = "POINT(33.3 38.4)"   # Tuz Gölü - Aksaray arası, GUNW çerçevesinin içi
TRACK = "_172_A_"                    # Vertex'te seçtiğin GUNW'nin yörüngesi
TARIH_1, TARIH_2 = "20260906", "20260918"

sonuc = asf.search(dataset="NISAR", processingLevel=["GCOV", "GUNW"],
                   intersectsWith=ALAN, start="2026-09-05", end="2026-09-19")
urunler = []
for r in sonuc:
    ad = r.properties.get("sceneName") or r.properties.get("fileID")
    urunler.append((ad, r.properties.get("url")))
    print(ad)
print(len(urunler), "ürün bulundu")

def tip(ad): return ad.split("_")[3]
def anahtar(ad):          # aynı track+frame'i gösteren kısım (cycle ve tarih hariç)
    t = ad.split("_"); return "_".join(t[5:8])

gunw = [u for u in urunler if tip(u[0]) == "GUNW" and TRACK in u[0] and TARIH_1 in u[0] and TARIH_2 in u[0]]
gcov = [u for u in urunler if tip(u[0]) == "GCOV" and TRACK in u[0]]
cift = None
for k in {anahtar(u[0]) for u in gcov}:
    once  = [u for u in gcov if anahtar(u[0]) == k and TARIH_1 in u[0]]
    sonra = [u for u in gcov if anahtar(u[0]) == k and TARIH_2 in u[0]]
    if once and sonra:
        cift = (once[0], sonra[0]); break

print("\nSeçilen GUNW :", gunw[0][0] if gunw else "BULUNAMADI")
print("Seçilen GCOV1:", cift[0][0] if cift else "BULUNAMADI")
print("Seçilen GCOV2:", cift[1][0] if cift else "BULUNAMADI")

def indir(url):
    if not url:
        return None
    yol = os.path.join("veri", url.split("/")[-1].split("?")[0])
    if not os.path.exists(yol):
        print("İndiriliyor:", os.path.basename(yol))
        asf.download_url(url=url, path="veri", session=session)
    print("Hazır:", os.path.basename(yol), f"({os.path.getsize(yol)/1e6:.0f} MB)")
    return yol

f_once  = indir(cift[0][1]) if cift else None
f_sonra = indir(cift[1][1]) if cift else None
f_gunw  = indir(gunw[0][1]) if gunw else None

In [ ]:
# Yardımcı fonksiyonlar (sadece çalıştır, değiştirmene gerek yok)
import h5py, numpy as np
import matplotlib.pyplot as plt

def agaci_yazdir(yol, max_satir=60):
    """Dosyanın içindeki veri katmanlarını listeler (hata ayıklamak için)."""
    satirlar = []
    with h5py.File(yol, "r") as f:
        def gez(ad, obj):
            if isinstance(obj, h5py.Dataset) and obj.ndim == 2:
                satirlar.append(f"{ad}  {obj.shape}")
        f["science"].visititems(gez)
    print("\n".join(satirlar[:max_satir]))

def bul(f, isim, tercih=("frequencyA",)):
    """Adı `isim` ile biten veri katmanının yolunu bulur."""
    adaylar = []
    f.visititems(lambda ad, o: adaylar.append(ad) if isinstance(o, h5py.Dataset) and ad.split("/")[-1] == isim else None)
    for t in tercih:
        for a in adaylar:
            if t in a:
                return a
    return adaylar[0] if adaylar else None

def katman_oku(yol, isimler, adim):
    """Görüntüyü + koordinatları okur. `adim` ile seyrekleştirir (bellek için)."""
    with h5py.File(yol, "r") as f:
        for isim in isimler:
            p = bul(f, isim)
            if p:
                break
        else:
            raise ValueError(f"{isimler} bulunamadı. agaci_yazdir() ile dosyaya bak.")
        grup = p.rsplit("/", 1)[0]
        # koordinatlar aynı grupta ya da bir üst grupta olur
        g = grup
        while "xCoordinates" not in f[g]:
            g = g.rsplit("/", 1)[0]
        x, y = f[g]["xCoordinates"][:], f[g]["yCoordinates"][:]
        veri = f[p][::adim, ::adim].astype("float32")
        return p, veri, x[::adim], y[::adim]

In [ ]:
# 4) GCOV: önce / sonra / fark
assert f_once and f_sonra, "GCOV çifti bulunamadı, 3. hücrenin çıktısını kontrol et."
ADIM = 8          # büyütürsen daha hızlı ama daha düşük çözünürlük
KATMAN = ["HHHH", "VVVV", "HVHV"]   # ilk bulunanı kullanır

p1, a, x1, y1 = katman_oku(f_once,  KATMAN, ADIM)
p2, b, x2, y2 = katman_oku(f_sonra, KATMAN, ADIM)
print("Kullanılan katman:", p1)

# İki görüntünün ortak alanını bul (aynı frame olsa da birkaç piksel kayabilir)
xl, xh = max(x1.min(), x2.min()), min(x1.max(), x2.max())
yl, yh = max(y1.min(), y2.min()), min(y1.max(), y2.max())
def kes(img, x, y):
    ix = np.where((x >= xl) & (x <= xh))[0]; iy = np.where((y >= yl) & (y <= yh))[0]
    return img[iy[0]:iy[-1]+1, ix[0]:ix[-1]+1]
a, b = kes(a, x1, y1), kes(b, x2, y2)
h, w = min(a.shape[0], b.shape[0]), min(a.shape[1], b.shape[1])
a, b = a[:h, :w], b[:h, :w]

def db(v):
    v = np.where(v > 0, v, np.nan)
    return 10 * np.log10(v)
a_db, b_db = db(a), db(b)
fark = b_db - a_db
kapsam = [xl/1000, xh/1000, yl/1000, yh/1000]   # km (UTM)

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
for i, (img, baslik) in enumerate([(a_db, "Önce (dB)"), (b_db, "Sonra (dB)")]):
    im = ax[i].imshow(img, cmap="gray", vmin=-25, vmax=0, extent=kapsam)
    ax[i].set_title(baslik); plt.colorbar(im, ax=ax[i], shrink=0.7)
im = ax[2].imshow(fark, cmap="RdBu_r", vmin=-5, vmax=5, extent=kapsam)
ax[2].set_title("Fark: Sonra − Önce (dB)\nkırmızı = daha parlak, mavi = daha karanlık")
plt.colorbar(im, ax=ax[2], shrink=0.7)
for x_ in ax: x_.set_xlabel("Doğu (km, UTM)"); x_.set_ylabel("Kuzey (km, UTM)")
plt.tight_layout(); plt.savefig("gcov_fark.png", dpi=150); plt.show()
print("Kaydedildi: gcov_fark.png (soldaki dosya panelinden indirebilirsin)")

In [ ]:
# 5) GUNW: zemin hareketi (LOS = uydunun bakış doğrultusunda)
if f_gunw:
    ADIM_G = 2
    p, faz, xg, yg = katman_oku(f_gunw, ["unwrappedPhase"], ADIM_G)
    print("Kullanılan katman:", p)
    with h5py.File(f_gunw, "r") as f:
        cc = bul(f, "connectedComponents")
        maske = f[cc][::ADIM_G, ::ADIM_G] if cc else None
        fr = bul(f, "centerFrequency")
        frekans = float(np.ravel(f[fr][()])[0]) if fr else 1.2575e9
    dalga_boyu = 299792458.0 / frekans   # L-band ≈ 24 cm
    faz = np.where((faz == 0) | ~np.isfinite(faz), np.nan, faz)
    if maske is not None and maske.shape == faz.shape:
        faz = np.where(maske == 0, np.nan, faz)          # güvenilmez pikselleri at
    yer_degistirme_cm = -faz * dalga_boyu / (4 * np.pi) * 100
    yer_degistirme_cm -= np.nanmedian(yer_degistirme_cm)  # ortancaya göre referansla

    sinir = np.nanpercentile(np.abs(yer_degistirme_cm), 98)
    plt.figure(figsize=(9, 8))
    plt.imshow(yer_degistirme_cm, cmap="RdBu_r", vmin=-sinir, vmax=sinir,
               extent=[xg.min()/1000, xg.max()/1000, yg.min()/1000, yg.max()/1000])
    plt.colorbar(label="LOS yer değiştirme (cm)")
    plt.title(f"GUNW – zemin hareketi (λ = {dalga_boyu*100:.1f} cm)")
    plt.xlabel("Doğu (km, UTM)"); plt.ylabel("Kuzey (km, UTM)")
    plt.savefig("gunw_hareket.png", dpi=150); plt.show()
    print("Not: işaret konvansiyonu (çökme + mı − mi) NISAR dokümanından teyit edilmeli.")
else:
    print("GUNW linki girilmedi, bu adım atlandı.")

## Hipotez doğrulama
**Hipotez:** 6–18 Eylül arasında yağmur yağdı → toprak nemlendi → kara parladı, Tuz Gölü karardı.

- **6a:** Şehirler de parladıysa fark kalibrasyondan olabilir. Şehirler sabit, tarlalar parladıysa nem hipotezi güçlenir.
- **6b:** NISAR'ın kendi toprak nemi ürünü (SME2) nemin arttığını gösteriyor mu?
- **6c:** NASA POWER yağış verisi: arada yağmur yağmış mı?

In [ ]:
# 6a) Şehir kontrolü: binalar yağmurla pek değişmez
from pyproj import Transformer

with h5py.File(f_once, "r") as f:
    p = bul(f, "projection")
    epsg = None
    if p:
        v = f[p][()]
        try:
            epsg = int(np.ravel(v)[0])
        except Exception:
            pass
        epsg = epsg if epsg and epsg > 1000 else int(f[p].attrs.get("epsg_code", 32636))
    epsg = epsg or 32636
print("Projeksiyon: EPSG", epsg)
tr = Transformer.from_crs(4326, epsg, always_xy=True)

# 4. hücredeki kırpılmış görüntünün koordinatları
xa = x1[(x1 >= xl) & (x1 <= xh)][:w]
ya = y1[(y1 >= yl) & (y1 <= yh)][:h]

yerler = {"Aksaray (şehir)": (34.03, 38.37), "Konya (şehir)": (32.49, 37.87),
          "Kulu (şehir)": (33.08, 39.09), "Tuz Gölü": (33.40, 38.75)}
YARICAP = 2000  # metre
print(f"\nTüm görüntünün ortanca farkı: {np.nanmedian(fark):+.2f} dB\n")
for ad, (lon, lat) in yerler.items():
    X, Y = tr.transform(lon, lat)
    ix = np.where(np.abs(xa - X) < YARICAP)[0]; iy = np.where(np.abs(ya - Y) < YARICAP)[0]
    if len(ix) == 0 or len(iy) == 0:
        print(f"{ad:18s}: görüntünün dışında"); continue
    blok = fark[iy[0]:iy[-1]+1, ix[0]:ix[-1]+1]
    print(f"{ad:18s}: {np.nanmedian(blok):+.2f} dB")
print("\nYorum: şehirler ~0 dB, genel ortanca +2/+3 dB ise → nem hipotezi güçlü."
      "\n       şehirler de ~genel ortanca kadar arttıysa → kalibrasyon farkı olabilir.")

In [ ]:
# 6b) NISAR toprak nemi (SME2) – Eylül tarihleri
sm = asf.search(dataset="NISAR", processingLevel=["SME2"], intersectsWith=ALAN,
                start="2026-09-01", end="2026-09-25")
sm_list = sorted([((r.properties.get("sceneName") or r.properties.get("fileID")),
                   r.properties.get("url"), r.properties.get("startTime")) for r in sm],
                 key=lambda t: t[2])
ayni_track = [t for t in sm_list if TRACK in t[0]]
secilen = (ayni_track or sm_list)[:4]
print(len(sm_list), "SME2 ürünü bulundu, kullanılanlar:")
for t in secilen: print("  ", t[2], t[0])

def nem_katmani(yol):
    adaylar = []
    with h5py.File(yol, "r") as f:
        f.visititems(lambda ad, o: adaylar.append(ad) if isinstance(o, h5py.Dataset)
                     and o.ndim == 2 and "soilmoisture" in ad.replace("_", "").lower() else None)
        if not adaylar:
            return None, None
        ad = adaylar[0]
        return ad, f[ad][::2, ::2].astype("float32")

tarihler, ortalamalar = [], []
for ad, url, t in secilen:
    yol = indir(url)
    katman, v = nem_katmani(yol)
    if katman is None:
        print("Nem katmanı bulunamadı:", os.path.basename(yol), "→ agaci_yazdir() ile bak"); continue
    gecerli = v[(v > 0) & (v < 1)]
    tarihler.append(t[:10]); ortalamalar.append(float(np.nanmean(gecerli)) if gecerli.size else np.nan)
    print(f"{t[:10]}  ({katman.split('/')[-1]}): ortalama nem = {ortalamalar[-1]:.3f} m³/m³")

if tarihler:
    plt.figure(figsize=(7, 3.5)); plt.plot(tarihler, ortalamalar, "o-")
    plt.ylabel("Toprak nemi (m³/m³)"); plt.title("NISAR SME2 – bölge ortalaması")
    plt.grid(alpha=.3); plt.tight_layout(); plt.savefig("sme2_nem.png", dpi=150); plt.show()

In [ ]:
# 6c) NASA POWER günlük yağış (giriş gerekmez)
import requests, datetime as dt
lon, lat = 33.3, 38.4
url = ("https://power.larc.nasa.gov/api/temporal/daily/point?parameters=PRECTOTCORR"
       f"&community=AG&longitude={lon}&latitude={lat}&start=20260901&end=20260920&format=JSON")
veri = requests.get(url, timeout=60).json()["properties"]["parameter"]["PRECTOTCORR"]
gunler = [dt.datetime.strptime(k, "%Y%m%d") for k in veri]
mm = [max(v, 0) if v != -999 else np.nan for v in veri.values()]

plt.figure(figsize=(9, 3.5))
plt.bar(gunler, mm, color="steelblue")
for d, et in [(dt.datetime(2026, 9, 6), "Önce"), (dt.datetime(2026, 9, 18), "Sonra")]:
    plt.axvline(d, color="red", ls="--"); plt.text(d, max(np.nanmax(mm), 1) * 0.9, " " + et, color="red")
plt.ylabel("Yağış (mm/gün)"); plt.title("NASA POWER – Tuz Gölü / Aksaray çevresi")
plt.tight_layout(); plt.savefig("yagis.png", dpi=150); plt.show()
ara = [m for g, m in zip(gunler, mm) if dt.datetime(2026, 9, 6) <= g < dt.datetime(2026, 9, 18)]
print(f"6–17 Eylül toplam yağış: {np.nansum(ara):.1f} mm  (görüntüler sabah ~05:53'te çekildi)")

## 7) Teslim paketi
Gürültü filtresi + ±3 dB değişim haritası + km² hesabı, sonra web sitesi ve slaytlar için
tüm görselleri ve rakamları **teslim.zip** olarak indirir. 4, 6a, 6b, 6c hücrelerinden sonra çalıştır.

In [ ]:
# 7) Gürültü filtresi, değişim haritası ve teslim paketi
import json, zipfile, shutil
from scipy.ndimage import uniform_filter
os.makedirs("teslim", exist_ok=True)

ESIK_DB = 3.0      # ±3 dB = güçte 2 kat değişim
PENCERE = 5        # benek (speckle) filtresi penceresi

def filtrele(guc):
    g = np.where(np.isfinite(guc) & (guc > 0), guc, np.nan)
    gecerli = np.isfinite(g)
    ort = uniform_filter(np.nan_to_num(g), PENCERE) / np.maximum(uniform_filter(gecerli.astype(float), PENCERE), 1e-6)
    return np.where(gecerli, ort, np.nan)

a_f, b_f = filtrele(a), filtrele(b)
a_fdb, b_fdb = db(a_f), db(b_f)
fark_f = b_fdb - a_fdb
degisim = np.zeros_like(fark_f, dtype=np.int8)
degisim[fark_f >  ESIK_DB] =  1
degisim[fark_f < -ESIK_DB] = -1
degisim = np.where(np.isfinite(fark_f), degisim, 0)

piksel_m2 = abs(xa[1] - xa[0]) * abs(ya[1] - ya[0])
gecerli_km2 = np.isfinite(fark_f).sum() * piksel_m2 / 1e6
parlayan_km2 = (degisim == 1).sum() * piksel_m2 / 1e6
kararan_km2  = (degisim == -1).sum() * piksel_m2 / 1e6
print(f"Piksel: {np.sqrt(piksel_m2):.0f} m | Analiz alanı: {gecerli_km2:,.0f} km²")
print(f"> +{ESIK_DB} dB parlayan: {parlayan_km2:,.0f} km² | < -{ESIK_DB} dB kararan: {kararan_km2:,.0f} km²")

# Gürültü ölçümü: filtre öncesi/sonrası fark std (tüm alan)
std_once, std_sonra = float(np.nanstd(fark)), float(np.nanstd(fark_f))
print(f"Fark std: filtresiz {std_once:.2f} dB → filtreli {std_sonra:.2f} dB")

# --- Web sitesi için eksen/yazı içermeyen görüntüler (aynı kapsam) ---
plt.imsave("teslim/before.png", a_fdb, cmap="gray", vmin=-25, vmax=0)
plt.imsave("teslim/after.png",  b_fdb, cmap="gray", vmin=-25, vmax=0)
plt.imsave("teslim/diff.png",   fark_f, cmap="RdBu_r", vmin=-5, vmax=5)
renk = np.full(degisim.shape + (4,), 0.0); renk[..., 3] = 0
renk[degisim == 1]  = (0.84, 0.19, 0.15, 1)
renk[degisim == -1] = (0.13, 0.40, 0.67, 1)
plt.imsave("teslim/change.png", renk)

# --- 2x2 panel (slayt ve README için) ---
fig, ax = plt.subplots(2, 2, figsize=(13, 11))
ax[0,0].imshow(a_fdb, cmap="gray", vmin=-25, vmax=0, extent=kapsam); ax[0,0].set_title("NISAR GCOV HH – 6 Sep 2026 (dB)")
ax[0,1].imshow(b_fdb, cmap="gray", vmin=-25, vmax=0, extent=kapsam); ax[0,1].set_title("NISAR GCOV HH – 18 Sep 2026 (dB)")
im = ax[1,0].imshow(fark_f, cmap="RdBu_r", vmin=-5, vmax=5, extent=kapsam); ax[1,0].set_title("Log-ratio (after − before, dB)")
plt.colorbar(im, ax=ax[1,0], shrink=0.75)
ax[1,1].imshow(np.full(degisim.shape, 0.92), cmap="gray", vmin=0, vmax=1, extent=kapsam)
ax[1,1].imshow(renk, extent=kapsam); ax[1,1].set_title(f"Change map (±{ESIK_DB:.0f} dB): red brighter, blue darker")
for x_ in ax.ravel(): x_.set_xlabel("Easting (km, UTM 36N)"); x_.set_ylabel("Northing (km)")
plt.suptitle("Konya Plain / Lake Tuz – NISAR L-band surface change", fontsize=14)
plt.tight_layout(); plt.savefig("teslim/panel_2x2.png", dpi=150); plt.show()

# --- Görüntü köşelerinin enlem/boylamı (web haritası için) ---
ters = Transformer.from_crs(epsg, 4326, always_xy=True)
kose = {k: ters.transform(X, Y) for k, (X, Y) in
        {"nw": (xl, yh), "ne": (xh, yh), "se": (xh, yl), "sw": (xl, yl)}.items()}

# --- Şehir kontrolü (filtreli) ---
sehir = {}
for ad, (lon, lat) in yerler.items():
    X, Y = tr.transform(lon, lat)
    ix = np.where(np.abs(xa - X) < YARICAP)[0]; iy = np.where(np.abs(ya - Y) < YARICAP)[0]
    if len(ix) and len(iy):
        sehir[ad] = round(float(np.nanmedian(fark_f[iy[0]:iy[-1]+1, ix[0]:ix[-1]+1])), 2)

g = globals()
rakamlar = {
    "aoi": "Konya Plain / Lake Tuz, Türkiye",
    "dates": ["2026-09-06T02:53:33Z", "2026-09-18T02:53:33Z"],
    "products": {"gcov_before": os.path.basename(f_once), "gcov_after": os.path.basename(f_sonra),
                 "gunw": os.path.basename(f_gunw) if f_gunw else None},
    "layer": p1, "epsg": int(epsg), "corners_lonlat": kose,
    "pixel_m": round(float(np.sqrt(piksel_m2)), 1), "analysed_km2": round(gecerli_km2),
    "median_change_db": round(float(np.nanmedian(fark_f)), 2),
    "threshold_db": ESIK_DB, "brighter_km2": round(parlayan_km2), "darker_km2": round(kararan_km2),
    "noise_std_db": {"unfiltered": round(std_once, 2), "filtered": round(std_sonra, 2)},
    "city_check_db": sehir,
    "rain_mm": ({d.strftime("%Y-%m-%d"): (None if np.isnan(m) else round(float(m), 1))
                 for d, m in zip(g["gunler"], g["mm"])} if "mm" in g else None),
    "sme2_mean": (dict(zip(g["tarihler"], [round(v, 3) for v in g["ortalamalar"]])) if "ortalamalar" in g else None),
}
json.dump(rakamlar, open("teslim/numbers.json", "w"), indent=2, ensure_ascii=False)
print(json.dumps(rakamlar, indent=2, ensure_ascii=False))

for f_ in ["gcov_fark.png", "gunw_hareket.png", "yagis.png", "sme2_nem.png"]:
    if os.path.exists(f_):
        shutil.copy(f_, f"teslim/{f_}")
with zipfile.ZipFile("teslim.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f_ in sorted(os.listdir("teslim")):
        z.write(f"teslim/{f_}", f_)
print("teslim.zip hazır:", sorted(os.listdir("teslim")))
try:
    from google.colab import files; files.download("teslim.zip")
except Exception:
    print("Soldaki dosya panelinden teslim.zip'i indir.")

In [ ]:
# Bir şey ters giderse: dosyanın içinde hangi katmanlar var?
# agaci_yazdir(f_once)
# agaci_yazdir(f_gunw)